# 为什么只检测到两个 Starlink VHF 事件

本 Notebook 汇总论文第4章的轨道可见性与轨道演化分析，用于：

1. 根据原始文件 `acquisition metadata` 重建五个观测时段，并用两次实测事件间隔外推潜在发射时刻；
2. 用 STARLINK-32838 的 SpaceX-E SupGP 检查这些预测时刻相对21CMA E08的地平几何；
3. 比较两起已观测事件在 SupGP 与当天 Active TLE 下的几何和运动学结果。

所有派生表格和图件均在本 Notebook 中重新生成。周期保留 `USE_MEASURED_CADENCE = True`，即使用两起事件实测得到的约99.987 s间隔，而不是改用理论100 s。


## 运行准备


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import csv
import importlib.util
import json
import math
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

%matplotlib inline

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'project_config.py').is_file():
            return candidate
    raise FileNotFoundError('Run from this repository or one of its subdirectories')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_config import DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, file_start_utc

OUTPUT_DIR = RESULTS_DIR / '4_orbital_context'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OBSERVATION_WINDOWS = DATA_DIR / 'observation' / 'observation_windows.csv'
EVENT_METADATA = DATA_DIR / 'observation' / 'event_files.csv'

print('Output directory:', OUTPUT_DIR)
print('Observation metadata:', OBSERVATION_WINDOWS)


## 1. 基于acquisition metadata 和约100 s重复周期的观测覆盖预测

以2200数据集中两起已确认事件的前导起点为锚点，默认使用两者的 acquisition metadata 时轴间隔作为重复周期，也可改为标称100 s。随后向前和向后逐周期外推，列出五个观测窗口内的预计时刻、文件号和文件内秒数。

这是基于“稳定周期”假设的确定性预测，不是检出结果，也不是置信区间。the acquisition UTC may contain recorder or clock offsets；随外推周期数增加，真实发射时刻也可能因 cadence 抖动而逐渐偏离。

In [ ]:
# Build the cadence sequence from fixed acquisition metadata.
NOMINAL_CADENCE_SECONDS = 100.0
USE_MEASURED_CADENCE = True
KNOWN_CADENCE_EVENTS = (
    {'event_id': 'Event 1', 'dataset': '202504152200',
     'file_number': 0, 'file_time_s': 5.330904,
     'filename': '20250415-2200-0.dat'},
    {'event_id': 'Event 2', 'dataset': '202504152200',
     'file_number': 12, 'file_time_s': 2.930004,
     'filename': '20250415-2200-12.dat'},
)
FILE_DURATION_SECONDS = 8_192_000_000 / 2 / 480_000_000

window_table = pd.read_csv(OBSERVATION_WINDOWS)
window_table['window_start_utc'] = pd.to_datetime(window_table['window_start_utc'], utc=True)
window_table['window_stop_utc'] = pd.to_datetime(window_table['window_stop_utc'], utc=True)
DATASETS = window_table['dataset'].astype(str).tolist()
observation_window_rows = []
records_by_dataset = {}
for row in window_table.itertuples(index=False):
    start = row.window_start_utc.to_pydatetime()
    stop = row.window_stop_utc.to_pydatetime()
    observation_window_rows.append({
        'window': row.window, 'dataset': str(row.dataset),
        'first_file_number': int(row.first_file_number),
        'last_file_number': int(row.last_file_number),
        'window_start_utc': start, 'window_stop_utc': stop,
        'window_span_s': (stop - start).total_seconds(),
        'recorded_file_duration_s': float(row.recorded_duration_s),
    })
    records_by_dataset[str(row.dataset)] = [
        {
            'dataset': str(row.dataset), 'file_number': number,
            'duration_s': FILE_DURATION_SECONDS,
            'start_utc': start + timedelta(seconds=number * FILE_DURATION_SECONDS),
            'stop_utc': start + timedelta(seconds=(number + 1) * FILE_DURATION_SECONDS),
        }
        for number in range(int(row.first_file_number), int(row.last_file_number) + 1)
    ]

event_file_table = pd.read_csv(EVENT_METADATA)
event_file_table['file_start_utc'] = pd.to_datetime(event_file_table['file_start_utc'], utc=True)
observed_event_rows = []
for event in KNOWN_CADENCE_EVENTS:
    metadata_row = event_file_table.loc[event_file_table['filename'].eq(event['filename'])].iloc[0]
    event_utc = metadata_row['file_start_utc'].to_pydatetime() + timedelta(seconds=event['file_time_s'])
    observed_event_rows.append({**event, 'event_utc': event_utc})

observed_event_rows.sort(key=lambda row: row['event_utc'])
measured_cadence_seconds = (
    observed_event_rows[1]['event_utc'] - observed_event_rows[0]['event_utc']
).total_seconds()
prediction_cadence_seconds = measured_cadence_seconds if USE_MEASURED_CADENCE else NOMINAL_CADENCE_SECONDS
observation_windows = pd.DataFrame(observation_window_rows)
observed_events_utc = pd.DataFrame(observed_event_rows)
observation_windows.to_csv(OUTPUT_DIR / 'observation_windows.csv', index=False)
observed_events_utc.to_csv(OUTPUT_DIR / 'observed_event_times.csv', index=False)
print(f'Measured cadence: {measured_cadence_seconds:.6f} s')
display(observation_windows)
display(observed_events_utc)


In [ ]:
# 16.2 从Event 1锚点按cadence向前、向后外推，并映射到原始文件
anchor_utc = observed_event_rows[0]['event_utc']


def _locate_prediction_in_file(dataset, predicted_utc):
    """Locate one UTC prediction in an actually recorded DAT interval."""
    for record in records_by_dataset[dataset]:
        file_start_utc, file_stop_utc = record['start_utc'], record['stop_utc']
        if file_start_utc <= predicted_utc <= file_stop_utc:
            return record, (predicted_utc - file_start_utc).total_seconds()
    return None, None


prediction_rows = []
for window in observation_window_rows:
    start_offset_s = (window['window_start_utc'] - anchor_utc).total_seconds()
    stop_offset_s = (window['window_stop_utc'] - anchor_utc).total_seconds()
    first_cycle = math.ceil(start_offset_s / prediction_cadence_seconds)
    last_cycle = math.floor(stop_offset_s / prediction_cadence_seconds)
    for cycle_index in range(first_cycle, last_cycle + 1):
        predicted_utc = anchor_utc + timedelta(
            seconds=cycle_index * prediction_cadence_seconds
        )
        record, file_time_s = _locate_prediction_in_file(
            window['dataset'], predicted_utc
        )
        prediction_rows.append({
            'dataset': window['dataset'],
            'cycle_index_from_event1': cycle_index,
            'predicted_utc': predicted_utc,
            'cadence_s': prediction_cadence_seconds,
            'inside_recorded_file': record is not None,
            'predicted_file_number': (
                int(record['file_number']) if record is not None else pd.NA
            ),
            'predicted_file_time_s': (
                float(file_time_s) if file_time_s is not None else np.nan
            ),
            'prediction_status': (
                'covered_by_DAT' if record is not None else 'between_recorded_files'
            ),
        })

cadence_predictions = pd.DataFrame(prediction_rows)
cadence_predictions.to_csv(
    OUTPUT_DIR / 'cadence_predictions.csv', index=False
)
prediction_counts = (
    cadence_predictions.groupby(['dataset', 'prediction_status'])
    .size().unstack(fill_value=0)
)
display(prediction_counts)
display(cadence_predictions)
print('Saved:', OUTPUT_DIR / 'cadence_predictions.csv')

In [ ]:
# 16.3 将五个观测窗口分行绘图
# 灰线是观测覆盖，蓝点是周期预测，红色五角星是已观测数据包。
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

# 可在这里集中修改图例、标题和横坐标文字。
PLOT_TEXT = {
    'observation_span': 'observation span',
    'cadence_prediction': 'cadence prediction in DAT',
    'observed_packet': 'observed packet',
    'gap_prediction': 'prediction in an inter-file gap',
    'x_axis': 'UTC from acquisition metadata',
    'title': 'Predicted Starlink VHF cadence opportunities ({cadence:.6f} s)',
}


def _format_prediction_utc(value):
    # 四舍五入显示到0.1 s；完整精度仍保存在cadence_predictions.csv。
    rounded = pd.Timestamp(value).round('100ms')
    return rounded.strftime('%H:%M:%S.%f')[:-5]

fig, axes = plt.subplots(
    len(DATASETS), 1, figsize=(15, 2.3 * len(DATASETS)),
    constrained_layout=True, squeeze=False,
)
for axis, window in zip(axes[:, 0], observation_window_rows):
    dataset = window['dataset']
    rows = cadence_predictions[cadence_predictions['dataset'] == dataset]
    covered = rows[rows['inside_recorded_file']]
    uncovered = rows[~rows['inside_recorded_file']]
    axis.hlines(
        0.0, window['window_start_utc'], window['window_stop_utc'],
        color='#B8B8B8', linewidth=4.0, zorder=1,
    )
    axis.scatter(
        covered['predicted_utc'], np.zeros(len(covered)),
        s=52, marker='o', color='#2878B5', edgecolor='white',
        linewidth=0.6, zorder=3,
    )
    if len(uncovered):
        axis.scatter(
            uncovered['predicted_utc'], np.zeros(len(uncovered)),
            s=58, marker='x', color='#F39B7F', linewidth=1.4, zorder=3,
        )
    ordered_rows = rows.sort_values('predicted_utc').reset_index(drop=True)
    for label_index, row in ordered_rows.iterrows():
        if label_index == 0:
            horizontal_alignment, horizontal_offset = 'left', 3
        elif label_index == len(ordered_rows) - 1:
            horizontal_alignment, horizontal_offset = 'right', -3
        else:
            horizontal_alignment, horizontal_offset = 'center', 0
        axis.annotate(
            _format_prediction_utc(row['predicted_utc']),
            (row['predicted_utc'], 0.0),
            xytext=(horizontal_offset, -13),
            textcoords='offset points',
            ha=horizontal_alignment, va='top',
            fontsize=8.2, color='#555555', rotation=0,
        )
    known = observed_events_utc[observed_events_utc['dataset'] == dataset]
    if len(known):
        axis.scatter(
            known['event_utc'], np.zeros(len(known)),
            s=170, marker='*', color='#D73027', edgecolor='black',
            linewidth=0.7, zorder=4,
        )
    axis.set_xlim(window['window_start_utc'], window['window_stop_utc'])
    axis.set_ylim(-0.30, 0.30)
    axis.set_yticks([])
    axis.set_ylabel(dataset, rotation=0, ha='right', va='center')
    axis.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S', tz=timezone.utc))
    axis.grid(axis='x', alpha=0.25)
legend_handles = [
    Line2D([0], [0], color='#B8B8B8', linewidth=3.5,
           label=PLOT_TEXT['observation_span']),
    Line2D([0], [0], marker='o', linestyle='None', markersize=7,
           markerfacecolor='#2878B5', markeredgecolor='white',
           label=PLOT_TEXT['cadence_prediction']),
    Line2D([0], [0], marker='*', linestyle='None', markersize=12,
           markerfacecolor='#D73027', markeredgecolor='black',
           label=PLOT_TEXT['observed_packet']),
]
if bool((~cadence_predictions['inside_recorded_file']).any()):
    legend_handles.append(
        Line2D([0], [0], marker='x', linestyle='None', markersize=7,
               color='#F39B7F', label=PLOT_TEXT['gap_prediction'])
    )
axes[0, 0].legend(handles=legend_handles, loc='upper center', ncol=4, fontsize=9)
axes[-1, 0].set_xlabel(PLOT_TEXT['x_axis'])
fig.suptitle(
    PLOT_TEXT['title'].format(cadence=prediction_cadence_seconds),
    fontsize=14,
)
prediction_plot = OUTPUT_DIR / 'cadence_prediction_overview.png'
fig.savefig(prediction_plot, dpi=220, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(prediction_plot)))
print('Saved:', prediction_plot)

## 2. 用SpaceX-E SupGP检查周期预测点的地平几何

本节将第1节的周期预测点与STARLINK-32838（NORAD 62827）的CelesTrak `SpaceX-E` SupGP轨道元素交叉。`cadence_predictions['predicted_utc']`表示由Event 1前导起点外推的预测包起点。对每个预测起点，选择历元时间最近的SupGP记录，并用同一组轨道元素分别传播到 `t-1 s`、`t`、`t+1 s`，计算E08站心处的方位角、几何高度角、斜距和卫星星下点。这样得到的是预测起点±1 s的三点时间敏感性检查，不是统计置信区间。

三点全部满足 `elevation > 0 deg` 表示在所采样的±1 s范围内均处于理想WGS84几何地平面之上；三点全部不高于0°表示三个采样时刻均在地平下；若正负状态混合，则标为可能跨越几何地平。三点采样不是对区间内每一时刻的严格连续证明，但对2 s局部变化提供直接检查。它不包含21CMA周边山体遮挡、单站方向图/旁瓣增益、极化、发射机是否正在发射以及检测灵敏度。因此，“地平上”是可接收的必要条件之一，不是充分条件。

SupGP是对运营商星历的SGP4拟合，通常比同时段标准GP/TLE更适合本任务，但CSV中的 `RMS` 不是位置协方差或1-sigma误差，本节不据此构造置信区间。

In [ ]:
# 20.1 读取并验证STARLINK-32838 SupGP和E08站心坐标
import csv
from sgp4 import omm
from sgp4.api import Satrec
from skyfield.api import EarthSatellite, load, wgs84

SUPGP_CSV = DATA_DIR / 'orbital' / 'STARLINK-32838_SpaceX-E_SupGP.csv'
STATION_CENTER_CSV = RESULTS_DIR / '2_waterfall' / 'e08_station_center_wgs84.csv'
GEOMETRIC_HORIZON_DEG = 0.0
SUPGP_EPOCH_WARNING_HOURS = 12.0
CADENCE_START_WINDOW_HALF_WIDTH_SECONDS = 1.0
CADENCE_WINDOW_SAMPLE_OFFSETS_SECONDS = (-1.0, 0.0, 1.0)

with SUPGP_CSV.open(newline='') as handle:
    raw_supgp_rows = list(csv.DictReader(handle))
if not raw_supgp_rows:
    raise ValueError(f'No SupGP rows in {SUPGP_CSV}')
if {row['NORAD_CAT_ID'] for row in raw_supgp_rows} != {'62827'}:
    raise ValueError('SupGP file must contain only NORAD 62827')
if {row['DATA_SOURCE'] for row in raw_supgp_rows} != {'SpaceX-E'}:
    raise ValueError('Expected CelesTrak SpaceX-E SupGP rows')

supgp_entries = sorted(
    [
        {
            'raw': row,
            'epoch_utc': datetime.fromisoformat(row['EPOCH']).replace(
                tzinfo=timezone.utc
            ),
        }
        for row in raw_supgp_rows
    ],
    key=lambda entry: entry['epoch_utc'],
)

station_center_table = pd.read_csv(STATION_CENTER_CSV)
required_station_columns = {
    'station', 'longitude_deg_east', 'latitude_deg_north',
    'ellipsoidal_height_m',
}
if required_station_columns - set(station_center_table.columns):
    raise ValueError('E08 station-center CSV is missing required columns')
if (
    len(station_center_table) != 1
    or str(station_center_table.iloc[0]['station']).upper() != 'E08'
):
    raise ValueError('Expected exactly one E08 station-center row')
station_row = station_center_table.iloc[0]
site_latitude_deg = float(station_row['latitude_deg_north'])
site_longitude_deg = float(station_row['longitude_deg_east'])
site_height_m = float(station_row['ellipsoidal_height_m'])

supgp_timescale = load.timescale()
e08_observer = wgs84.latlon(
    latitude_degrees=site_latitude_deg,
    longitude_degrees=site_longitude_deg,
    elevation_m=site_height_m,
)

print('SupGP:', SUPGP_CSV)
print(
    'SupGP coverage:', supgp_entries[0]['epoch_utc'].isoformat(),
    'to', supgp_entries[-1]['epoch_utc'].isoformat(),
)
print(
    f'E08: lat={site_latitude_deg:.9f} deg, '
    f'lon={site_longitude_deg:.9f} deg, h={site_height_m:.3f} m'
)

In [ ]:
# 20.2 对第19节每个周期预测起点及其前后1 s计算地平几何
if 'cadence_predictions' not in globals():
    raise RuntimeError('Run Sections 19.1 and 19.2 first.')


def _nearest_supgp_entry(utc):
    return min(
        supgp_entries,
        key=lambda entry: abs((entry['epoch_utc'] - utc).total_seconds()),
    )


def _satellite_from_supgp_entry(entry):
    satrec = Satrec()
    omm.initialize(satrec, entry['raw'])
    return EarthSatellite.from_satrec(satrec, supgp_timescale)


def _sample_topocentric_geometry(satellite, entry, sample_utc):
    skyfield_time = supgp_timescale.from_datetime(sample_utc)
    topocentric = (satellite - e08_observer).at(skyfield_time)
    altitude, azimuth, distance = topocentric.altaz()
    subpoint = wgs84.subpoint(satellite.at(skyfield_time))
    epoch_offset_hours = (
        sample_utc - entry['epoch_utc']
    ).total_seconds() / 3600.0
    return {
        'geometry_sample_utc': sample_utc,
        'supgp_epoch_utc': entry['epoch_utc'],
        'supgp_epoch_offset_hours': epoch_offset_hours,
        'supgp_epoch_abs_separation_hours': abs(epoch_offset_hours),
        'supgp_fit_rms': float(entry['raw']['RMS']),
        'azimuth_deg': float(azimuth.degrees),
        'elevation_deg': float(altitude.degrees),
        'slant_range_km': float(distance.km),
        'above_geometric_horizon': bool(
            altitude.degrees > GEOMETRIC_HORIZON_DEG
        ),
        'subpoint_latitude_deg': float(subpoint.latitude.degrees),
        'subpoint_longitude_deg': float(subpoint.longitude.degrees),
        'subpoint_height_km': float(subpoint.elevation.km),
    }


def _shortest_azimuth_change_deg(start_deg, stop_deg):
    return float((stop_deg - start_deg + 180.0) % 360.0 - 180.0)


visibility_rows = []  # 中心时刻，兼容原有20.3绘图。
visibility_window_sample_rows = []  # 每个预测事件三行：-1、0、+1 s。
visibility_window_summary_rows = []  # 每个预测事件一行的变化摘要。
for _, prediction in cadence_predictions.iterrows():
    predicted_utc = pd.Timestamp(prediction['predicted_utc']).to_pydatetime()
    if predicted_utc.tzinfo is None:
        predicted_utc = predicted_utc.replace(tzinfo=timezone.utc)
    entry = _nearest_supgp_entry(predicted_utc)
    satellite = _satellite_from_supgp_entry(entry)
    observed_label = ''
    for event in observed_event_rows:
        if abs((predicted_utc - event['event_utc']).total_seconds()) < 0.05:
            observed_label = event['event_id']
            break
    sampled_geometry = {}
    for sample_offset_s in CADENCE_WINDOW_SAMPLE_OFFSETS_SECONDS:
        sample_utc = predicted_utc + timedelta(seconds=sample_offset_s)
        geometry = _sample_topocentric_geometry(
            satellite, entry, sample_utc
        )
        sampled_geometry[float(sample_offset_s)] = geometry
        visibility_window_sample_rows.append({
            **prediction.to_dict(),
            'observed_event_id': observed_label,
            'geometry_sample_offset_s': float(sample_offset_s),
            **geometry,
        })

    minus_geometry = sampled_geometry[-1.0]
    center_geometry = sampled_geometry[0.0]
    plus_geometry = sampled_geometry[1.0]
    visibility_rows.append({
        **prediction.to_dict(),
        'observed_event_id': observed_label,
        **center_geometry,
    })
    sampled_elevations = [
        sampled_geometry[offset]['elevation_deg']
        for offset in CADENCE_WINDOW_SAMPLE_OFFSETS_SECONDS
    ]
    sampled_above = [
        sampled_geometry[offset]['above_geometric_horizon']
        for offset in CADENCE_WINDOW_SAMPLE_OFFSETS_SECONDS
    ]
    if all(sampled_above):
        sampled_horizon_status = 'all_three_above_horizon'
    elif not any(sampled_above):
        sampled_horizon_status = 'all_three_at_or_below_horizon'
    else:
        sampled_horizon_status = 'mixed_possible_horizon_crossing'
    visibility_window_summary_rows.append({
        **prediction.to_dict(),
        'observed_event_id': observed_label,
        'window_half_width_s': CADENCE_START_WINDOW_HALF_WIDTH_SECONDS,
        'supgp_epoch_utc': entry['epoch_utc'],
        'elevation_minus_1s_deg': minus_geometry['elevation_deg'],
        'elevation_center_deg': center_geometry['elevation_deg'],
        'elevation_plus_1s_deg': plus_geometry['elevation_deg'],
        'elevation_change_minus1_to_plus1_deg': (
            plus_geometry['elevation_deg'] - minus_geometry['elevation_deg']
        ),
        'minimum_sampled_elevation_deg': min(sampled_elevations),
        'maximum_sampled_elevation_deg': max(sampled_elevations),
        'azimuth_minus_1s_deg': minus_geometry['azimuth_deg'],
        'azimuth_center_deg': center_geometry['azimuth_deg'],
        'azimuth_plus_1s_deg': plus_geometry['azimuth_deg'],
        'azimuth_change_minus1_to_plus1_deg': (
            _shortest_azimuth_change_deg(
                minus_geometry['azimuth_deg'], plus_geometry['azimuth_deg']
            )
        ),
        'sampled_horizon_status': sampled_horizon_status,
    })

supgp_cadence_visibility = pd.DataFrame(visibility_rows)
supgp_cadence_visibility_window_samples = pd.DataFrame(
    visibility_window_sample_rows
)
supgp_cadence_visibility_window_summary = pd.DataFrame(
    visibility_window_summary_rows
)
if (
    supgp_cadence_visibility_window_samples[
        'supgp_epoch_abs_separation_hours'
    ]
    .max() > SUPGP_EPOCH_WARNING_HOURS
):
    print('WARNING: at least one prediction is more than 12 h from its SupGP epoch.')
supgp_cadence_visibility.to_csv(
    OUTPUT_DIR / 'supgp_cadence_visibility.csv', index=False
)
supgp_cadence_visibility_window_samples.to_csv(
    OUTPUT_DIR / 'supgp_cadence_visibility_window_samples.csv',
    index=False,
)
supgp_cadence_visibility_window_summary.to_csv(
    OUTPUT_DIR / 'supgp_cadence_visibility_window_summary.csv',
    index=False,
)

visibility_summary = (
    supgp_cadence_visibility.groupby('dataset', as_index=False)
    .agg(
        predicted_opportunities=('predicted_utc', 'count'),
        above_horizon_count=('above_geometric_horizon', 'sum'),
        minimum_elevation_deg=('elevation_deg', 'min'),
        maximum_elevation_deg=('elevation_deg', 'max'),
        minimum_slant_range_km=('slant_range_km', 'min'),
        maximum_supgp_epoch_separation_h=('supgp_epoch_abs_separation_hours', 'max'),
    )
)
window_dataset_summary = (
    supgp_cadence_visibility_window_summary.groupby(
        'dataset', as_index=False
    )
    .agg(
        all_three_above_horizon_count=(
            'sampled_horizon_status',
            lambda values: int((values == 'all_three_above_horizon').sum()),
        ),
        mixed_horizon_count=(
            'sampled_horizon_status',
            lambda values: int((values == 'mixed_possible_horizon_crossing').sum()),
        ),
        all_three_at_or_below_horizon_count=(
            'sampled_horizon_status',
            lambda values: int((values == 'all_three_at_or_below_horizon').sum()),
        ),
        maximum_abs_elevation_change_over_2s_deg=(
            'elevation_change_minus1_to_plus1_deg',
            lambda values: float(values.abs().max()),
        ),
        maximum_abs_azimuth_change_over_2s_deg=(
            'azimuth_change_minus1_to_plus1_deg',
            lambda values: float(values.abs().max()),
        ),
    )
)
visibility_summary = visibility_summary.merge(
    window_dataset_summary, on='dataset', how='left'
)
visibility_summary.to_csv(
    OUTPUT_DIR / 'supgp_cadence_visibility_summary.csv', index=False
)
display(visibility_summary)
display(supgp_cadence_visibility_window_summary)
print('Saved:', OUTPUT_DIR / 'supgp_cadence_visibility.csv')
print(
    'Saved:',
    OUTPUT_DIR / 'supgp_cadence_visibility_window_samples.csv',
)
print(
    'Saved:',
    OUTPUT_DIR / 'supgp_cadence_visibility_window_summary.csv',
)

In [ ]:
# 20.3 分数据集绘制预测中心点的几何高度角
fig, axes = plt.subplots(
    len(DATASETS), 1, figsize=(15, 2.5 * len(DATASETS)),
    constrained_layout=True, squeeze=False,
)
for axis, dataset in zip(axes[:, 0], DATASETS):
    rows = (
        supgp_cadence_visibility[
            supgp_cadence_visibility['dataset'] == dataset
        ]
        .sort_values('predicted_utc')
    )
    visible = rows['above_geometric_horizon']
    axis.axhline(0.0, color='black', linewidth=1.0, alpha=0.7)
    axis.plot(
        rows['predicted_utc'], rows['elevation_deg'],
        color='0.65', linewidth=1.2, zorder=1,
    )
    axis.scatter(
        rows.loc[visible, 'predicted_utc'], rows.loc[visible, 'elevation_deg'],
        s=55, color='#2A9D55', label='above geometric horizon', zorder=3,
    )
    axis.scatter(
        rows.loc[~visible, 'predicted_utc'], rows.loc[~visible, 'elevation_deg'],
        s=45, color='0.55', label='below geometric horizon', zorder=3,
    )
    observed = rows[rows['observed_event_id'] != '']
    if len(observed):
        axis.scatter(
            observed['predicted_utc'], observed['elevation_deg'],
            s=180, marker='*', color='#D73027', edgecolor='black',
            linewidth=0.7, label='observed packet', zorder=4,
        )
    for _, row in rows.iterrows():
        if bool(row['inside_recorded_file']):
            axis.annotate(
                f"{int(row['predicted_file_number'])}:{row['predicted_file_time_s']:.2f}s",
                (row['predicted_utc'], row['elevation_deg']),
                xytext=(0, 7), textcoords='offset points',
                ha='center', fontsize=8, rotation=30,
            )
    axis.set_ylabel(f'{dataset}\nElevation (deg)')
    axis.xaxis.set_major_formatter(
        mdates.DateFormatter('%H:%M:%S', tz=timezone.utc)
    )
    axis.grid(alpha=0.25)
axes[0, 0].legend(loc='best', fontsize=9)
axes[-1, 0].set_xlabel('Predicted UTC from the acquisition metadata cadence model')
fig.suptitle('STARLINK-32838 SupGP elevation at cadence-predicted times', fontsize=14)
visibility_plot = OUTPUT_DIR / 'supgp_cadence_visibility.png'
fig.savefig(visibility_plot, dpi=220, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(visibility_plot)))
print('Saved:', visibility_plot)

In [ ]:
# 20.4 在21CMA E08本地ENU单位天球上展示五个观测窗口与周期预测点
# 坐标定义：x=East，y=North，z=Up；球半径只表示天空方向，不表示卫星距离。
# 每个观测窗口固定使用距离其中点最近的一组SupGP，避免轨迹内部切换轨道元素。
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

SKY_TRACK_STEP_SECONDS = 1.0
SKY_TRACK_COLORS = {
    '202504151940': '#4C78A8',
    '202504152100': '#F58518',
    '202504152200': '#54A24B',
    '202504160036': '#B279A2',
    '202504160735': "#9D1984",
}
# 下列参数集中控制图中线宽、点大小和观察视角。
SKY_TRACK_LINEWIDTH = 2
SKY_PREDICTION_POINT_SIZE = 34
SKY_OBSERVED_EVENT_SIZE = 190
SKY_HORIZON_LINEWIDTH = 1.5
SKY_SPHERE_GRID_LINEWIDTH = 0.65
SKY_GROUND_GRID_LINEWIDTH = 0.55
SKY_SPHERE_LIMB_LINEWIDTH = 1.1
SKY_STATION_POINT_SIZE = 75
SKY_ALTITUDE_LABEL_AZIMUTH_DEG = 205
SKY_VIEW_ELEVATION_DEG = 12
SKY_VIEW_AZIMUTH_DEG = -55


def _enu_unit_from_altaz(azimuth_deg, elevation_deg):
    """Convert north-through-east azimuth and elevation to ENU unit vectors."""
    azimuth_rad = np.deg2rad(np.asarray(azimuth_deg, dtype=float))
    elevation_rad = np.deg2rad(np.asarray(elevation_deg, dtype=float))
    horizontal_radius = np.cos(elevation_rad)
    east = horizontal_radius * np.sin(azimuth_rad)
    north = horizontal_radius * np.cos(azimuth_rad)
    up = np.sin(elevation_rad)
    return east, north, up


def _utc_window_label(start_utc, stop_utc):
    start = pd.Timestamp(start_utc)
    stop = pd.Timestamp(stop_utc)
    return f"{start.strftime('%m-%d %H:%M:%S')}–{stop.strftime('%H:%M:%S')} UTC"


# 连续传播五个acquisition metadata观测窗口；每隔1 s保存一个方向和几何量。
sky_track_rows = []
sky_track_epoch_rows = []
for window in observation_window_rows:
    dataset = window['dataset']
    window_start_utc = pd.Timestamp(window['window_start_utc']).to_pydatetime()
    window_stop_utc = pd.Timestamp(window['window_stop_utc']).to_pydatetime()
    if window_start_utc.tzinfo is None:
        window_start_utc = window_start_utc.replace(tzinfo=timezone.utc)
    if window_stop_utc.tzinfo is None:
        window_stop_utc = window_stop_utc.replace(tzinfo=timezone.utc)
    window_midpoint_utc = window_start_utc + (window_stop_utc - window_start_utc) / 2
    entry = _nearest_supgp_entry(window_midpoint_utc)
    satellite = _satellite_from_supgp_entry(entry)

    window_span_s = (window_stop_utc - window_start_utc).total_seconds()
    sample_offsets_s = np.arange(
        0.0, window_span_s, SKY_TRACK_STEP_SECONDS, dtype=float
    )
    if not len(sample_offsets_s) or sample_offsets_s[-1] < window_span_s:
        sample_offsets_s = np.append(sample_offsets_s, window_span_s)
    sample_times_utc = [
        window_start_utc + timedelta(seconds=float(offset_s))
        for offset_s in sample_offsets_s
    ]
    skyfield_times = supgp_timescale.from_datetimes(sample_times_utc)
    topocentric = (satellite - e08_observer).at(skyfield_times)
    altitude, azimuth, distance = topocentric.altaz()
    east, north, up = _enu_unit_from_altaz(
        azimuth.degrees, altitude.degrees
    )

    for row_index, sample_utc in enumerate(sample_times_utc):
        sky_track_rows.append({
            'dataset': dataset,
            'sample_utc': sample_utc,
            'seconds_from_window_start': float(sample_offsets_s[row_index]),
            'supgp_epoch_utc': entry['epoch_utc'],
            'azimuth_deg': float(azimuth.degrees[row_index]),
            'elevation_deg': float(altitude.degrees[row_index]),
            'slant_range_km': float(distance.km[row_index]),
            'above_geometric_horizon': bool(altitude.degrees[row_index] > 0.0),
            'enu_unit_east': float(east[row_index]),
            'enu_unit_north': float(north[row_index]),
            'enu_unit_up': float(up[row_index]),
        })
    sky_track_epoch_rows.append({
        'dataset': dataset,
        'window_start_utc': window_start_utc,
        'window_midpoint_utc': window_midpoint_utc,
        'window_stop_utc': window_stop_utc,
        'supgp_epoch_utc': entry['epoch_utc'],
        'supgp_epoch_offset_at_midpoint_hours': (
            window_midpoint_utc - entry['epoch_utc']
        ).total_seconds() / 3600.0,
        'trajectory_sample_interval_s': SKY_TRACK_STEP_SECONDS,
        'trajectory_sample_count': len(sample_times_utc),
    })

supgp_observation_window_sky_tracks = pd.DataFrame(sky_track_rows)
supgp_observation_window_track_epochs = pd.DataFrame(sky_track_epoch_rows)
sky_tracks_csv = OUTPUT_DIR / 'supgp_observation_window_sky_tracks.csv'
sky_track_epochs_csv = OUTPUT_DIR / 'supgp_observation_window_track_epochs.csv'
supgp_observation_window_sky_tracks.to_csv(sky_tracks_csv, index=False)
supgp_observation_window_track_epochs.to_csv(sky_track_epochs_csv, index=False)
display(supgp_observation_window_track_epochs)

# 绘制透明单位天球、z=0地平面和本地东—北—天坐标。
fig = plt.figure(figsize=(8, 6.6))
axis = fig.add_subplot(111, projection='3d')
# 让天球尽可能占据画布，图例仅放在角落的空白区域。
axis.set_position([0.01, 0.01, 0.98, 0.98])
axis.set_proj_type('ortho')

sphere_azimuth = np.linspace(0.0, 2.0 * np.pi, 361)
# 这些纬圈正式称为等高度圈（circles of equal altitude / almucantars）。
altitude_circle_degrees = (-60, -30, 0, 30, 60)
for elevation_deg in altitude_circle_degrees:
    elevation_rad = np.deg2rad(elevation_deg)
    circle_radius = np.cos(elevation_rad)
    circle_up = np.sin(elevation_rad)
    is_horizon = elevation_deg == 0
    axis.plot(
        circle_radius * np.sin(sphere_azimuth),
        circle_radius * np.cos(sphere_azimuth),
        np.full_like(sphere_azimuth, circle_up),
        color='0.25' if is_horizon else ('#8EBAD9' if elevation_deg > 0 else '0.72'),
        linestyle='-' if elevation_deg >= 0 else '--',
        linewidth=(
            SKY_HORIZON_LINEWIDTH if is_horizon
            else SKY_SPHERE_GRID_LINEWIDTH
        ),
        alpha=0.85 if is_horizon else 0.55,
        zorder=1,
    )
    label_azimuth_rad = np.deg2rad(SKY_ALTITUDE_LABEL_AZIMUTH_DEG)
    axis.text(
        circle_radius * np.sin(label_azimuth_rad),
        circle_radius * np.cos(label_azimuth_rad),
        circle_up,
        f'{elevation_deg:+d}°' if elevation_deg else '0°',
        fontsize=9.5, color='0.38', ha='center', va='center', zorder=3,
    )
sphere_elevation = np.linspace(-0.5 * np.pi, 0.5 * np.pi, 181)
for azimuth_deg in range(0, 360, 30):
    azimuth_rad = np.deg2rad(azimuth_deg)
    horizontal_radius = np.cos(sphere_elevation)
    axis.plot(
        horizontal_radius * np.sin(azimuth_rad),
        horizontal_radius * np.cos(azimuth_rad),
        np.sin(sphere_elevation),
        color='#8EBAD9', linewidth=SKY_SPHERE_GRID_LINEWIDTH,
        alpha=0.38, zorder=1,
    )
# 画出当前观察方向下单位球的真实外轮廓，防止把前表面轨迹误读成离开球面。
view_elevation_rad = np.deg2rad(SKY_VIEW_ELEVATION_DEG)
view_azimuth_rad = np.deg2rad(SKY_VIEW_AZIMUTH_DEG)
view_direction = np.array([
    np.cos(view_elevation_rad) * np.cos(view_azimuth_rad),
    np.cos(view_elevation_rad) * np.sin(view_azimuth_rad),
    np.sin(view_elevation_rad),
])
limb_basis_1 = np.cross(view_direction, np.array([0.0, 0.0, 1.0]))
limb_basis_1 /= np.linalg.norm(limb_basis_1)
limb_basis_2 = np.cross(view_direction, limb_basis_1)
limb_points = (
    limb_basis_1[:, None] * np.cos(sphere_azimuth)[None, :]
    + limb_basis_2[:, None] * np.sin(sphere_azimuth)[None, :]
)
axis.plot(
    limb_points[0], limb_points[1], limb_points[2],
    color='0.38', linewidth=SKY_SPHERE_LIMB_LINEWIDTH,
    alpha=0.75, zorder=2,
)
# 用线框表示水平面，避免Matplotlib的半透明曲面遮住地平线下轨迹。
for ground_circle_radius in (0.25, 0.50, 0.75):
    axis.plot(
        ground_circle_radius * np.sin(sphere_azimuth),
        ground_circle_radius * np.cos(sphere_azimuth),
        np.zeros_like(sphere_azimuth),
        color='0.70', linewidth=SKY_GROUND_GRID_LINEWIDTH,
        alpha=0.38, zorder=0,
    )
for ground_azimuth_deg in range(0, 360, 30):
    ground_azimuth_rad = np.deg2rad(ground_azimuth_deg)
    axis.plot(
        [0.0, np.sin(ground_azimuth_rad)],
        [0.0, np.cos(ground_azimuth_rad)],
        [0.0, 0.0],
        color='0.70', linewidth=SKY_GROUND_GRID_LINEWIDTH,
        alpha=0.38, zorder=0,
    )

# 每个观测窗口均画为一条连续实线；球面与z=0地平面负责表达地平线上下。
track_legend_handles = []
for window_index, window in enumerate(observation_window_rows, start=1):
    dataset = window['dataset']
    color = SKY_TRACK_COLORS[dataset]
    rows = (
        supgp_observation_window_sky_tracks[
            supgp_observation_window_sky_tracks['dataset'] == dataset
        ].sort_values('sample_utc')
    )
    east = rows['enu_unit_east'].to_numpy(dtype=float)
    north = rows['enu_unit_north'].to_numpy(dtype=float)
    up = rows['enu_unit_up'].to_numpy(dtype=float)
    axis.plot(
        east, north, up,
        color=color, linewidth=SKY_TRACK_LINEWIDTH,
        linestyle='-', alpha=0.95, zorder=5,
    )
    track_legend_handles.append(
        Line2D(
            [0], [0], color=color, linewidth=2.5,
            label=f'W{window_index}',
        )
    )

# 蓝点只表示未实测的周期预测起点；两起实测事件只画红色五角星。
observed_mask = (
    supgp_cadence_visibility['observed_event_id']
    .fillna('').astype(str).ne('')
)
prediction_rows = supgp_cadence_visibility.loc[~observed_mask]
prediction_east, prediction_north, prediction_up = _enu_unit_from_altaz(
    prediction_rows['azimuth_deg'],
    prediction_rows['elevation_deg'],
)
axis.scatter(
    prediction_east, prediction_north, prediction_up,
    s=SKY_PREDICTION_POINT_SIZE, marker='o', color='#2878B5',
    edgecolor='white', linewidth=0.5,
    depthshade=False, zorder=6,
)
observed_rows = supgp_cadence_visibility.loc[observed_mask]
observed_east, observed_north, observed_up = _enu_unit_from_altaz(
    observed_rows['azimuth_deg'], observed_rows['elevation_deg']
)
axis.scatter(
    observed_east, observed_north, observed_up,
    s=SKY_OBSERVED_EVENT_SIZE, marker='*', color='#D73027',
    edgecolor='black', linewidth=0.8,
    depthshade=False, zorder=8,
)
for east, north, up, event_id, elevation_deg in zip(
    observed_east, observed_north, observed_up,
    observed_rows['observed_event_id'], observed_rows['elevation_deg'],
):
    # Event 2单独向右上方移动文字，避免与轨迹和五角星拥挤。
    if event_id == 'Event 2':
        text_east = east + 0.10
        text_north = north
        text_up = up + 0.08
    else:
        text_east = east
        text_north = north
        text_up = up + 0.055
    axis.text(
        text_east, text_north, text_up,
        f'{event_id}\n{elevation_deg:.2f}°',
        fontsize=10, ha='center',
    )

# 球心红点表示21CMA E08观测站。
axis.scatter(
    [0.0], [0.0], [0.02], s=SKY_STATION_POINT_SIZE, marker='o',
    color='#D73027', edgecolor='black', linewidth=0.7,
    depthshade=False, zorder=8,
)
axis.text(0.045, 0.045, 0.055, '21CMA E08', fontsize=11, weight='bold')

for label, east, north in (
    ('N', 0.0, 1.08), ('E', 1.08, 0.0),
    ('S', 0.0, -1.08), ('W', -1.08, 0.0),
):
    axis.text(east, north, 0.02, label, fontsize=12, weight='bold', ha='center')
axis.text(0.0, 0.0, 1.08, 'Zenith', fontsize=11, ha='center')

axis.set_xlim(-1.12, 1.12)
axis.set_ylim(-1.12, 1.12)
axis.set_zlim(-1.05, 1.12)
# zoom只放大三维天球的显示，不改变任何轨道或角度数据。
axis.set_box_aspect((1.0, 1.0, 1.0), zoom=1.7)
axis.view_init(
    elev=SKY_VIEW_ELEVATION_DEG, azim=SKY_VIEW_AZIMUTH_DEG
)
axis.set_axis_off()
track_legend = axis.legend(
    handles=track_legend_handles, title='Observing windows',
    loc='upper left', bbox_to_anchor=(0.00, 0.99), ncol=2,
    fontsize=7.5, title_fontsize=8.0, frameon=True,
    handlelength=1.5, handletextpad=0.5, labelspacing=0.3,
    columnspacing=0.8, borderpad=0.35,
)
axis.add_artist(track_legend)
symbol_legend_handles = [
    Line2D([0], [0], marker='o', linestyle='None', markersize=5,
           markerfacecolor='#2878B5', markeredgecolor='white',
           label='Predicted start'),
    Line2D([0], [0], marker='*', linestyle='None', markersize=8,
           markerfacecolor='#D73027', markeredgecolor='black',
           label='Observed event'),
]
axis.legend(
    handles=symbol_legend_handles, loc='upper right',
    bbox_to_anchor=(1.00, 0.99), fontsize=7.5, frameon=True,
    handletextpad=0.5, labelspacing=0.3, borderpad=0.35,
)
local_sky_plot = OUTPUT_DIR / 'supgp_local_sky_sphere.png'
local_sky_pdf = OUTPUT_DIR / 'supgp_local_sky_sphere.pdf'
fig.savefig(local_sky_plot, dpi=260, bbox_inches='tight')
fig.savefig(local_sky_pdf, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(local_sky_plot)))
print('Saved:', sky_tracks_csv)
print('Saved:', sky_track_epochs_csv)
print('Saved:', local_sky_plot)
print('Saved:', local_sky_pdf)

In [ ]:
# 上图地下亮点连线标角度

# 20.4a 用标准方位角—高度角极坐标补充展示2200轨迹与两个事件
# 极坐标半径是天顶距 r=90°-elevation；因此相同高度角必然落在近似相同半径上。
# 本单元只读取20.4已经生成的数据，不修改20.4三维天球图。
POLAR_DATASET = '202504152200'
POLAR_TRACK_LINEWIDTH = 2.6
POLAR_PREDICTION_POINT_SIZE = 34
POLAR_EVENT_MARKER_SIZE = 155
POLAR_EQUAL_ALTITUDE_LINEWIDTH = 1.2

if 'supgp_observation_window_sky_tracks' not in globals():
    raise RuntimeError('Run Section 20.4 first.')
if 'supgp_cadence_visibility' not in globals():
    raise RuntimeError('Run Section 20.2 first.')

polar_track = (
    supgp_observation_window_sky_tracks[
        (supgp_observation_window_sky_tracks['dataset'] == POLAR_DATASET)
        & (supgp_observation_window_sky_tracks['elevation_deg'] >= 0.0)
    ]
    .sort_values('sample_utc')
)
polar_predictions = (
    supgp_cadence_visibility[
        (supgp_cadence_visibility['dataset'] == POLAR_DATASET)
        & (supgp_cadence_visibility['elevation_deg'] >= 0.0)
    ]
    .copy()
)
polar_observed_mask = (
    polar_predictions['observed_event_id']
    .fillna('').astype(str).ne('')
)
polar_observed = polar_predictions.loc[polar_observed_mask]
polar_unobserved_predictions = polar_predictions.loc[~polar_observed_mask]
if len(polar_observed) != 2:
    raise ValueError(
        f'Expected two observed events above the horizon; found {len(polar_observed)}'
    )

fig = plt.figure(figsize=(5.4, 5.4))
axis = fig.add_subplot(111, projection='polar')
axis.set_theta_zero_location('N')
axis.set_theta_direction(-1)

track_theta = np.deg2rad(polar_track['azimuth_deg'].to_numpy(float))
track_zenith_distance_deg = (
    90.0 - polar_track['elevation_deg'].to_numpy(float)
)
axis.plot(
    track_theta, track_zenith_distance_deg,
    color=SKY_TRACK_COLORS[POLAR_DATASET],
    linewidth=POLAR_TRACK_LINEWIDTH, zorder=3,
    label='2200 observation track',
)

prediction_theta = np.deg2rad(
    polar_unobserved_predictions['azimuth_deg'].to_numpy(float)
)
prediction_radius = (
    90.0 - polar_unobserved_predictions['elevation_deg'].to_numpy(float)
)
axis.scatter(
    prediction_theta, prediction_radius,
    s=POLAR_PREDICTION_POINT_SIZE, marker='o',
    color='#2878B5', edgecolor='white', linewidth=0.5, zorder=5,
    label='cadence-predicted start',
)

observed_theta = np.deg2rad(polar_observed['azimuth_deg'].to_numpy(float))
observed_radius = 90.0 - polar_observed['elevation_deg'].to_numpy(float)
axis.scatter(
    observed_theta, observed_radius,
    s=POLAR_EVENT_MARKER_SIZE, marker='*',
    color='#D73027', edgecolor='black', linewidth=0.7, zorder=7,
    label='observed event',
)
for theta, radius, (_, row) in zip(
    observed_theta, observed_radius, polar_observed.iterrows()
):
    axis.annotate(
        f"{row['observed_event_id']}\n{row['elevation_deg']:.1f}°",
        (theta, radius), xytext=(0, 10), textcoords='offset points',
        ha='center', va='bottom', fontsize=8.5, zorder=8,
    )

# 用两事件平均高度角画一条辅助等高度圈，直接说明二者高度几乎相同。
mean_event_elevation_deg = float(polar_observed['elevation_deg'].mean())
event_altitude_radius = 90.0 - mean_event_elevation_deg
full_theta = np.linspace(0.0, 2.0 * np.pi, 721)
axis.plot(
    full_theta, np.full_like(full_theta, event_altitude_radius),
    color='#D73027', linewidth=POLAR_EQUAL_ALTITUDE_LINEWIDTH,
    linestyle=(0, (3, 2)), alpha=0.50, zorder=2,
)
axis.text(
    np.deg2rad(205.0), event_altitude_radius,
    f'event altitude ≈ {mean_event_elevation_deg:.1f}°',
    fontsize=8, color='#A32020', ha='center', va='bottom',
    rotation=25, rotation_mode='anchor',
)

axis.set_ylim(0.0, 90.0)
axis.set_thetagrids(
    [0, 45, 90, 135, 180, 225, 270, 315],
    labels=['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'],
)
# 半径是天顶距；标签改写成对应高度角。
axis.set_rgrids(
    [0, 30, 60, 90], labels=['90°', '60°', '30°', '0°'], angle=225
)
axis.grid(color='0.75', linewidth=0.65, alpha=0.70)
axis.spines['polar'].set_linewidth(1.2)
axis.spines['polar'].set_color('0.25')
axis.set_title(
    'STARLINK-32838 above the 21CMA E08 horizon\n'
    'altitude–azimuth projection',
    fontsize=11.5, pad=18,
)
axis.legend(
    loc='lower center', bbox_to_anchor=(0.5, -0.20),
    ncol=3, fontsize=7.8, frameon=True,
)
polar_inset_plot = OUTPUT_DIR / 'supgp_2200_altaz_polar_inset.png'
fig.savefig(polar_inset_plot, dpi=260, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(polar_inset_plot)))
print(
    'Observed elevations:',
    ', '.join(
        f"{row['observed_event_id']}={row['elevation_deg']:.6f} deg"
        for _, row in polar_observed.iterrows()
    ),
)
print('Saved:', polar_inset_plot)

In [ ]:
# 20.5 在acquisition metadata的LoRa preamble UTC上比较SupGP与Active_0415 TLE
# 输出文字表格，不绘图。经纬度和高度是卫星WGS84星下点；方位角、高度角、
# 斜距和距离变化率均相对于21CMA E08站心。ECEF速度用于与包内状态保持同一坐标系。
import json
from skyfield.framelib import itrs

ACTIVE_0415_TLE_PATH = DATA_DIR / 'orbital' / 'Active_2025-04-15.tle'
TARGET_NORAD_ID = 62827
REFERENCE_RF_FREQUENCY_HZ = 137_055_000.0
LIGHT_SPEED_KM_S = 299_792.458

if 'observed_event_rows' not in globals():
    raise RuntimeError('Run Section 19.1 first to reconstruct the two preamble UTC times.')
if 'supgp_entries' not in globals() or 'e08_observer' not in globals():
    raise RuntimeError('Run Sections 20.1 and 20.2 first.')
if not ACTIVE_0415_TLE_PATH.is_file():
    raise FileNotFoundError(ACTIVE_0415_TLE_PATH)

active_catalogue = load.tle_file(str(ACTIVE_0415_TLE_PATH))
active_target_candidates = [
    satellite for satellite in active_catalogue
    if int(satellite.model.satnum) == TARGET_NORAD_ID
]
if len(active_target_candidates) != 1:
    raise ValueError(
        f'Expected one NORAD {TARGET_NORAD_ID} record in Active_0415.tle; '
        f'found {len(active_target_candidates)}'
    )
active_target = active_target_candidates[0]
active_target_epoch_utc = active_target.epoch.utc_datetime()


def _event_orbit_geometry_row(event_id, event_utc, orbit_source, satellite, orbit_epoch_utc):
    """Calculate one reproducible event/source geometry and state row."""
    skyfield_time = supgp_timescale.from_datetime(event_utc)
    satellite_state = satellite.at(skyfield_time)
    satellite_position_ecef, satellite_velocity_ecef = (
        satellite_state.frame_xyz_and_velocity(itrs)
    )
    station_state = e08_observer.at(skyfield_time)
    station_position_ecef, station_velocity_ecef = (
        station_state.frame_xyz_and_velocity(itrs)
    )
    relative_position_ecef_km = (
        satellite_position_ecef.km - station_position_ecef.km
    )
    relative_velocity_ecef_km_s = (
        satellite_velocity_ecef.km_per_s - station_velocity_ecef.km_per_s
    )
    geometric_range_km = float(np.linalg.norm(relative_position_ecef_km))
    range_rate_km_s = float(
        np.dot(relative_position_ecef_km, relative_velocity_ecef_km_s)
        / geometric_range_km
    )
    topocentric = (satellite - e08_observer).at(skyfield_time)
    altitude, azimuth, skyfield_distance = topocentric.altaz()
    subpoint = wgs84.subpoint(satellite_state)
    predicted_doppler_hz = (
        -REFERENCE_RF_FREQUENCY_HZ * range_rate_km_s / LIGHT_SPEED_KM_S
    )
    return {
        'event_id': event_id,
        'preamble_utc': event_utc.isoformat(),
        'orbit_source': orbit_source,
        'orbit_epoch_utc': orbit_epoch_utc.isoformat(),
        'orbit_age_at_event_hours': (
            event_utc - orbit_epoch_utc
        ).total_seconds() / 3600.0,
        'subpoint_latitude_deg_north': float(subpoint.latitude.degrees),
        'subpoint_longitude_deg_east': float(subpoint.longitude.degrees),
        'satellite_wgs84_height_km': float(subpoint.elevation.km),
        'azimuth_deg_north_through_east': float(azimuth.degrees),
        'geometric_elevation_deg': float(altitude.degrees),
        'slant_range_km': float(skyfield_distance.km),
        'range_rate_km_s_positive_receding': range_rate_km_s,
        'predicted_doppler_at_137p055MHz_hz': predicted_doppler_hz,
        'gcrs_speed_km_s': float(
            np.linalg.norm(satellite_state.velocity.km_per_s)
        ),
        'ecef_x_km': float(satellite_position_ecef.km[0]),
        'ecef_y_km': float(satellite_position_ecef.km[1]),
        'ecef_z_km': float(satellite_position_ecef.km[2]),
        'ecef_vx_km_s': float(satellite_velocity_ecef.km_per_s[0]),
        'ecef_vy_km_s': float(satellite_velocity_ecef.km_per_s[1]),
        'ecef_vz_km_s': float(satellite_velocity_ecef.km_per_s[2]),
        'ecef_speed_km_s': float(
            np.linalg.norm(satellite_velocity_ecef.km_per_s)
        ),
    }


event_orbit_rows = []
for event in sorted(observed_event_rows, key=lambda row: row['event_utc']):
    event_utc = pd.Timestamp(event['event_utc']).to_pydatetime()
    if event_utc.tzinfo is None:
        event_utc = event_utc.replace(tzinfo=timezone.utc)
    supgp_entry = _nearest_supgp_entry(event_utc)
    supgp_target = _satellite_from_supgp_entry(supgp_entry)
    event_orbit_rows.append(
        _event_orbit_geometry_row(
            event['event_id'], event_utc, 'CelesTrak SpaceX-E SupGP',
            supgp_target, supgp_entry['epoch_utc'],
        )
    )
    event_orbit_rows.append(
        _event_orbit_geometry_row(
            event['event_id'], event_utc, 'Active_0415.tle',
            active_target, active_target_epoch_utc,
        )
    )

event_orbit_comparison = pd.DataFrame(event_orbit_rows)
geometry_columns = [
    'event_id', 'preamble_utc', 'orbit_source', 'orbit_epoch_utc',
    'orbit_age_at_event_hours',
    'subpoint_latitude_deg_north', 'subpoint_longitude_deg_east',
    'satellite_wgs84_height_km', 'azimuth_deg_north_through_east',
    'geometric_elevation_deg', 'slant_range_km',
    'range_rate_km_s_positive_receding',
    'predicted_doppler_at_137p055MHz_hz',
    'gcrs_speed_km_s', 'ecef_speed_km_s',
]
state_columns = [
    'event_id', 'orbit_source', 'preamble_utc',
    'ecef_x_km', 'ecef_y_km', 'ecef_z_km',
    'ecef_vx_km_s', 'ecef_vy_km_s', 'ecef_vz_km_s', 'ecef_speed_km_s',
]
event_geometry_table = event_orbit_comparison[geometry_columns].copy()
event_state_vector_table = event_orbit_comparison[state_columns].copy()

# 差值方向固定为SupGP减Active_0415 TLE。
difference_columns = [
    'subpoint_latitude_deg_north', 'subpoint_longitude_deg_east',
    'satellite_wgs84_height_km', 'azimuth_deg_north_through_east',
    'geometric_elevation_deg', 'slant_range_km',
    'range_rate_km_s_positive_receding',
    'predicted_doppler_at_137p055MHz_hz',
    'gcrs_speed_km_s', 'ecef_speed_km_s',
]
difference_rows = []
for event_id, rows in event_orbit_comparison.groupby('event_id', sort=False):
    supgp_row = rows[rows['orbit_source'] == 'CelesTrak SpaceX-E SupGP'].iloc[0]
    active_row = rows[rows['orbit_source'] == 'Active_0415.tle'].iloc[0]
    difference_rows.append({
        'event_id': event_id,
        'difference_definition': 'SupGP minus Active_0415.tle',
        **{column: float(supgp_row[column] - active_row[column])
           for column in difference_columns},
    })
event_orbit_differences = pd.DataFrame(difference_rows)

# 补充诊断：读取Event 1包内GPS状态及XYZ/Vxyz拟合得到的最佳模型时间偏移。
# 这些偏移不是新的接收时刻；它们是对轨道解沿轨相位误差的经验校正。
ALIGNMENT_SUMMARY_PATH = (
    RESULTS_DIR / '3_satellite_identification'
    / 'supgp_validation_summary.json'
)
if not ALIGNMENT_SUMMARY_PATH.is_file():
    raise FileNotFoundError(ALIGNMENT_SUMMARY_PATH)
with ALIGNMENT_SUMMARY_PATH.open(encoding='utf-8') as handle:
    alignment_summary = json.load(handle)
active_tle_best_offset_s = float(
    alignment_summary['original_winner_tle']['best_offset_s']
)
supgp_best_offset_s = float(
    alignment_summary['selected_pre_event_supgp']['best_offset_s']
)
alignment_reference_utc = alignment_summary['event1_packet_state']['gps_state_utc']

# 检查偏移所对应的轨道解与本单元实际读取的轨道解一致。
alignment_active_epoch = datetime.fromisoformat(
    alignment_summary['original_winner_tle']['epoch_utc']
)
alignment_supgp_epoch = datetime.fromisoformat(
    alignment_summary['selected_pre_event_supgp']['epoch_utc']
)
if abs((active_target_epoch_utc - alignment_active_epoch).total_seconds()) > 1e-3:
    raise ValueError('Active TLE epoch does not match the fitted-offset solution')

best_offset_rows = []
for event in sorted(observed_event_rows, key=lambda row: row['event_utc']):
    reception_utc = pd.Timestamp(event['event_utc']).to_pydatetime()
    if reception_utc.tzinfo is None:
        reception_utc = reception_utc.replace(tzinfo=timezone.utc)
    supgp_entry = _nearest_supgp_entry(reception_utc)
    if abs((supgp_entry['epoch_utc'] - alignment_supgp_epoch).total_seconds()) > 1e-3:
        raise ValueError('Nearest SupGP epoch does not match the fitted-offset solution')
    corrected_solutions = [
        (
            'Active_0415.tle + best offset', active_target,
            active_target_epoch_utc, active_tle_best_offset_s,
        ),
        (
            'CelesTrak SpaceX-E SupGP + best offset',
            _satellite_from_supgp_entry(supgp_entry),
            supgp_entry['epoch_utc'], supgp_best_offset_s,
        ),
    ]
    for source, satellite, orbit_epoch_utc, best_offset_s in corrected_solutions:
        model_utc = reception_utc + timedelta(seconds=best_offset_s)
        model_row = _event_orbit_geometry_row(
            event['event_id'], model_utc, source, satellite, orbit_epoch_utc
        )
        best_offset_rows.append({
            'event_id': event['event_id'],
            'reception_preamble_utc': reception_utc.isoformat(),
            'orbit_source': source,
            'orbit_epoch_utc': orbit_epoch_utc.isoformat(),
            'best_offset_s_from_event1_state_match': best_offset_s,
            'model_utc_after_offset': model_utc.isoformat(),
            'orbit_age_at_model_time_hours': model_row['orbit_age_at_event_hours'],
            'subpoint_latitude_deg_north': model_row['subpoint_latitude_deg_north'],
            'subpoint_longitude_deg_east': model_row['subpoint_longitude_deg_east'],
            'satellite_wgs84_height_km': model_row['satellite_wgs84_height_km'],
            'azimuth_deg_north_through_east': model_row['azimuth_deg_north_through_east'],
            'geometric_elevation_deg': model_row['geometric_elevation_deg'],
            'slant_range_km': model_row['slant_range_km'],
            'range_rate_km_s_positive_receding': (
                model_row['range_rate_km_s_positive_receding']
            ),
            'predicted_doppler_at_137p055MHz_hz': (
                model_row['predicted_doppler_at_137p055MHz_hz']
            ),
            'gcrs_speed_km_s': model_row['gcrs_speed_km_s'],
            'ecef_speed_km_s': model_row['ecef_speed_km_s'],
        })
best_offset_corrected_geometry = pd.DataFrame(best_offset_rows)

geometry_csv = OUTPUT_DIR / 'event_geometry_supgp_vs_active_tle.csv'
state_csv = OUTPUT_DIR / 'event_state_vectors_supgp_vs_active_tle.csv'
difference_csv = OUTPUT_DIR / 'event_geometry_supgp_minus_active_tle.csv'
best_offset_csv = OUTPUT_DIR / 'event_geometry_best_offset_corrected_models.csv'
event_geometry_table.to_csv(geometry_csv, index=False)
event_state_vector_table.to_csv(state_csv, index=False)
event_orbit_differences.to_csv(difference_csv, index=False)
best_offset_corrected_geometry.to_csv(best_offset_csv, index=False)

print('Table 20.5a — event geometry (angles in deg; distance/height in km)')
display(event_geometry_table)
print('Table 20.5b — ECEF/ITRS state vectors (km and km/s)')
display(event_state_vector_table)
print('Table 20.5c — SupGP minus Active_0415.tle')
display(event_orbit_differences)
print(
    'Table 20.5d — best-offset-corrected model geometry; '
    'offsets fitted at Event 1 payload GPS-state UTC',
)
print('Alignment reference UTC:', alignment_reference_utc)
display(best_offset_corrected_geometry)
print('Saved:', geometry_csv)
print('Saved:', state_csv)
print('Saved:', difference_csv)
print('Saved:', best_offset_csv)

## 3. 五个观测窗口的全球星下点轨迹与22:00时段局部图


本节使用与第2节相同的 SpaceX-E SupGP 和 acquisition metadata 重建观测窗口，绘制两张二维星下点地图：左图给出五个观测窗口的全球轨迹，右图放大22:00 UTC观测窗口，并区分几何地平线上方的周期预测机会与两次实际检出事件。


图中星下点只表示卫星在地表的垂直投影；它不等于接收点，也不单独决定可探测性。周期预测点仍采用第1节的实测间隔（约99.987 s）。


In [ ]:
# 21.1 绘制全球星下点轨迹和22:00时段区域放大图
# 底图只使用Natural Earth Physical Coastline。
from matplotlib.lines import Line2D

COASTLINE_GEOJSON = DATA_DIR / 'maps' / 'ne_50m_coastline.geojson'
GROUND_TRACK_STEP_SECONDS = 1.0
FULL_GROUND_TRACK_STEP_SECONDS = 10.0
GROUND_TRACK_OUTPUT_STEM = OUTPUT_DIR / 'supgp_ground_track_coastline_only'
TARGET_MAP_DATASET = '202504152200'

if not COASTLINE_GEOJSON.is_file():
    raise FileNotFoundError(COASTLINE_GEOJSON)
if 'supgp_entries' not in globals() or 'e08_observer' not in globals():
    raise RuntimeError('Run Sections 20.1 and 20.2 first.')
if 'cadence_predictions' not in globals():
    raise RuntimeError('Run Section 19 first.')
if 'supgp_cadence_visibility' not in globals():
    raise RuntimeError('Run Section 20.2 first.')

with COASTLINE_GEOJSON.open(encoding='utf-8') as handle:
    coastline_geojson = json.load(handle)


def _iter_physical_coastlines(geometry):
    """Yield lon/lat arrays from Natural Earth physical coastline geometry."""
    geometry_type = geometry.get('type')
    coordinates = geometry.get('coordinates', [])
    if geometry_type == 'LineString':
        yield np.asarray(coordinates, dtype=float)
    elif geometry_type == 'MultiLineString':
        for line in coordinates:
            yield np.asarray(line, dtype=float)
    else:
        raise ValueError(f'Unexpected coastline geometry: {geometry_type}')


def _draw_coastlines(axis, *, color='#777777', linewidth=0.55):
    """Draw physical coastlines only; no administrative boundary data."""
    for feature in coastline_geojson['features']:
        for line in _iter_physical_coastlines(feature['geometry']):
            if len(line) >= 2:
                axis.plot(
                    line[:, 0], line[:, 1], color=color,
                    linewidth=linewidth, zorder=0,
                )


def _plot_wrapped_track(axis, longitude_deg, latitude_deg, **plot_kwargs):
    """Split a ground track at the ±180° seam before plotting."""
    longitude_deg = np.asarray(longitude_deg, dtype=float)
    latitude_deg = np.asarray(latitude_deg, dtype=float)
    breaks = np.where(np.abs(np.diff(longitude_deg)) > 180.0)[0] + 1
    for lon_part, lat_part in zip(
        np.split(longitude_deg, breaks),
        np.split(latitude_deg, breaks),
    ):
        if len(lon_part) >= 2:
            axis.plot(lon_part, lat_part, **plot_kwargs)


# 与20.4一致：每个acquisition metadata观测窗口固定使用距离窗口中点最近的一组SupGP。
ground_track_rows = []
for window in observation_window_rows:
    dataset = window['dataset']
    window_start_utc = pd.Timestamp(window['window_start_utc']).to_pydatetime()
    window_stop_utc = pd.Timestamp(window['window_stop_utc']).to_pydatetime()
    if window_start_utc.tzinfo is None:
        window_start_utc = window_start_utc.replace(tzinfo=timezone.utc)
    if window_stop_utc.tzinfo is None:
        window_stop_utc = window_stop_utc.replace(tzinfo=timezone.utc)
    midpoint_utc = window_start_utc + (window_stop_utc - window_start_utc) / 2
    entry = _nearest_supgp_entry(midpoint_utc)
    satellite = _satellite_from_supgp_entry(entry)

    span_seconds = (window_stop_utc - window_start_utc).total_seconds()
    offsets_seconds = np.arange(
        0.0, span_seconds, GROUND_TRACK_STEP_SECONDS, dtype=float
    )
    if not len(offsets_seconds) or offsets_seconds[-1] < span_seconds:
        offsets_seconds = np.append(offsets_seconds, span_seconds)
    sample_times_utc = [
        window_start_utc + timedelta(seconds=float(offset_seconds))
        for offset_seconds in offsets_seconds
    ]
    skyfield_times = supgp_timescale.from_datetimes(sample_times_utc)
    subpoints = wgs84.subpoint(satellite.at(skyfield_times))
    longitude_deg = np.asarray(subpoints.longitude.degrees, dtype=float)
    latitude_deg = np.asarray(subpoints.latitude.degrees, dtype=float)
    height_km = np.asarray(subpoints.elevation.km, dtype=float)

    for sample_utc, offset_seconds, lon, lat, height in zip(
        sample_times_utc,
        offsets_seconds,
        longitude_deg,
        latitude_deg,
        height_km,
    ):
        ground_track_rows.append({
            'dataset': dataset,
            'sample_utc': sample_utc,
            'seconds_from_window_start': float(offset_seconds),
            'supgp_epoch_utc': entry['epoch_utc'],
            'subpoint_longitude_deg': float(lon),
            'subpoint_latitude_deg': float(lat),
            'subpoint_height_km': float(height),
        })

ground_tracks = pd.DataFrame(ground_track_rows)
ground_tracks.to_csv(
    OUTPUT_DIR / 'supgp_observation_window_ground_tracks.csv', index=False
)

# 从第一组观测开始到最后一组结束，补算完整星下点轨迹作为上下文。
# 每个采样时刻使用历元最近的SupGP；不同SupGP记录之间分段传播和绘制。
full_track_start_utc = min(
    pd.Timestamp(window['window_start_utc']).to_pydatetime()
    for window in observation_window_rows
)
full_track_stop_utc = max(
    pd.Timestamp(window['window_stop_utc']).to_pydatetime()
    for window in observation_window_rows
)
if full_track_start_utc.tzinfo is None:
    full_track_start_utc = full_track_start_utc.replace(tzinfo=timezone.utc)
if full_track_stop_utc.tzinfo is None:
    full_track_stop_utc = full_track_stop_utc.replace(tzinfo=timezone.utc)
full_track_span_seconds = (
    full_track_stop_utc - full_track_start_utc
).total_seconds()
full_track_offsets_seconds = np.arange(
    0.0, full_track_span_seconds, FULL_GROUND_TRACK_STEP_SECONDS, dtype=float
)
if (
    not len(full_track_offsets_seconds)
    or full_track_offsets_seconds[-1] < full_track_span_seconds
):
    full_track_offsets_seconds = np.append(
        full_track_offsets_seconds, full_track_span_seconds
    )
full_track_times_utc = [
    full_track_start_utc + timedelta(seconds=float(offset_seconds))
    for offset_seconds in full_track_offsets_seconds
]
nearest_supgp_indices = np.asarray([
    min(
        range(len(supgp_entries)),
        key=lambda index: abs(
            (supgp_entries[index]['epoch_utc'] - sample_utc).total_seconds()
        ),
    )
    for sample_utc in full_track_times_utc
], dtype=int)
segment_breaks = np.where(np.diff(nearest_supgp_indices) != 0)[0] + 1
segment_index_groups = np.split(
    np.arange(len(full_track_times_utc), dtype=int), segment_breaks
)
full_ground_track_rows = []
for sample_indices in segment_index_groups:
    if not len(sample_indices):
        continue
    entry = supgp_entries[int(nearest_supgp_indices[sample_indices[0]])]
    satellite = _satellite_from_supgp_entry(entry)
    segment_times_utc = [full_track_times_utc[index] for index in sample_indices]
    segment_skyfield_times = supgp_timescale.from_datetimes(segment_times_utc)
    segment_subpoints = wgs84.subpoint(
        satellite.at(segment_skyfield_times)
    )
    for index, lon, lat, height in zip(
        sample_indices,
        np.asarray(segment_subpoints.longitude.degrees, dtype=float),
        np.asarray(segment_subpoints.latitude.degrees, dtype=float),
        np.asarray(segment_subpoints.elevation.km, dtype=float),
    ):
        full_ground_track_rows.append({
            'sample_utc': full_track_times_utc[index],
            'seconds_from_total_span_start': float(
                full_track_offsets_seconds[index]
            ),
            'supgp_epoch_utc': entry['epoch_utc'],
            'subpoint_longitude_deg': float(lon),
            'subpoint_latitude_deg': float(lat),
            'subpoint_height_km': float(height),
        })
full_ground_track = pd.DataFrame(full_ground_track_rows).sort_values(
    'sample_utc'
)
full_ground_track.to_csv(
    OUTPUT_DIR / 'supgp_full_span_ground_track.csv', index=False
)

fig, (global_axis, regional_axis) = plt.subplots(
    1, 2, figsize=(16.2, 6.1), constrained_layout=True,
    gridspec_kw={'width_ratios': [1.42, 1.0]},
)

for axis in (global_axis, regional_axis):
    axis.set_facecolor('#FAFCFE')
    _draw_coastlines(axis)
    axis.grid(color='#D4DCE4', linewidth=0.55, alpha=0.65)
    axis.set_xlabel('Longitude (°E)', fontsize=11)
    axis.set_ylabel('Latitude (°N)', fontsize=11)
    axis.tick_params(axis='both', which='major', labelsize=11)

# (a) 先画完整时间跨度的浅灰轨迹，再叠加五个acquisition metadata观测窗口。
for _, epoch_rows in full_ground_track.groupby(
    'supgp_epoch_utc', sort=False
):
    epoch_rows = epoch_rows.sort_values('sample_utc')
    _plot_wrapped_track(
        global_axis,
        epoch_rows['subpoint_longitude_deg'],
        epoch_rows['subpoint_latitude_deg'],
        color='#B7B7B7', linewidth=0.85, alpha=0.78, zorder=1,
    )

for dataset in DATASETS:
    rows = ground_tracks[ground_tracks['dataset'] == dataset]
    is_target = dataset == TARGET_MAP_DATASET
    _plot_wrapped_track(
        global_axis,
        rows['subpoint_longitude_deg'],
        rows['subpoint_latitude_deg'],
        color=('#D73027' if is_target else '#4C78A8'),
        linewidth=(2.8 if is_target else 1.8),
        alpha=(1.0 if is_target else 0.82),
        zorder=(4 if is_target else 2),
    )

global_axis.scatter(
    site_longitude_deg, site_latitude_deg,
    s=165, marker='*', color='#F2B701', edgecolor='black',
    linewidth=0.8, zorder=8,
)
global_axis.annotate(
    f'21CMA E08\n{site_longitude_deg:.2f}°E, {site_latitude_deg:.2f}°N',
    (site_longitude_deg, site_latitude_deg),
    xytext=(10, 8), textcoords='offset points',
    ha='left', va='bottom', fontsize=8.8,
)
global_axis.set_xlim(-180.0, 180.0)
global_axis.set_ylim(-70.0, 85.0)
global_axis.set_xticks(np.arange(-180, 181, 60))
global_axis.set_yticks(np.arange(-60, 91, 30))
global_axis.set_title(
    '(a) Sub-satellite-point tracks during the five observing windows',
    fontsize=13,
)

global_legend = [
    Line2D([0], [0], color='#B7B7B7', linewidth=0.85,
           label='Full sub-satellite-point track over the observation span'),
    Line2D([0], [0], color='#D73027', linewidth=2.8,
           label='14:00 UTC observing window'),
    Line2D([0], [0], color='#4C78A8', linewidth=1.8,
           label='Other observing windows'),
    Line2D([0], [0], marker='*', linestyle='none', markersize=12,
           markerfacecolor='#F2B701', markeredgecolor='black', label='21CMA E08'),
]
global_axis.legend(handles=global_legend, loc='lower left', fontsize=8.5)

# (b) 22:00 UTC轨迹及几何地平线上方的四个周期预测机会。
regional_track = ground_tracks[
    ground_tracks['dataset'] == TARGET_MAP_DATASET
].sort_values('sample_utc')
_plot_wrapped_track(
    regional_axis,
    regional_track['subpoint_longitude_deg'],
    regional_track['subpoint_latitude_deg'],
    color='#C44E52', linewidth=2.8, zorder=3,
)
regional_axis.scatter(
    site_longitude_deg, site_latitude_deg,
    s=175, marker='*', color='#F2B701', edgecolor='black',
    linewidth=0.8, zorder=8,
)
regional_axis.annotate(
    f'21CMA E08\n{site_longitude_deg:.2f}°E\n{site_latitude_deg:.2f}°N',
    (site_longitude_deg, site_latitude_deg),
    xytext=(-10, -20), textcoords='offset points',
    ha='right', va='top', fontsize=8.8,
)

visible_opportunities = (
    supgp_cadence_visibility[
        (supgp_cadence_visibility['dataset'] == TARGET_MAP_DATASET)
        & (supgp_cadence_visibility['above_geometric_horizon'])
    ]
    .sort_values('predicted_utc')
    .copy()
)
detected_mask = (
    visible_opportunities['observed_event_id']
    .fillna('').astype(str).ne('')
)
detected_events = visible_opportunities.loc[detected_mask]
undetected_opportunities = visible_opportunities.loc[~detected_mask]

regional_axis.scatter(
    undetected_opportunities['subpoint_longitude_deg'],
    undetected_opportunities['subpoint_latitude_deg'],
    s=75, marker='o', facecolor='white', edgecolor='#2878B5',
    linewidth=1.8, zorder=7,
)
regional_axis.scatter(
    detected_events['subpoint_longitude_deg'],
    detected_events['subpoint_latitude_deg'],
    s=165, marker='*', color='#D73027', edgecolor='black',
    linewidth=0.7, zorder=9,
)
observed_event_time_map = {
    str(event['event_id']): pd.Timestamp(event['event_utc'])
    for event in observed_event_rows
}
for _, row in visible_opportunities.iterrows():
    event_label = str(row['observed_event_id']).strip()
    label_time = observed_event_time_map.get(
        event_label, pd.Timestamp(row['predicted_utc'])
    )
    utc_label = label_time.round('100ms').strftime('%H:%M:%S.%f')[:-5]
    label = f'{event_label}\n{utc_label} UTC' if event_label else f'{utc_label} UTC'
    regional_axis.annotate(
        label,
        (row['subpoint_longitude_deg'], row['subpoint_latitude_deg']),
        xytext=(5, 6), textcoords='offset points',
        ha='left', va='bottom', fontsize=8.0,
    )

regional_axis.set_xlim(77.0, 135.0)
regional_axis.set_ylim(5.0, 53.0)
regional_axis.set_xticks(np.arange(80, 136, 10))
regional_axis.set_yticks(np.arange(10, 51, 10))
regional_axis.set_title(
    '(b) 14:00 UTC window and visible cadence opportunities',
    fontsize=13,
)
regional_legend = [
    Line2D([0], [0], color='#C44E52', linewidth=2.8,
           label='14:00 UTC sub-satellite-point track'),
    Line2D([0], [0], marker='o', linestyle='none', markersize=7,
           markerfacecolor='white', markeredgewidth=1.6,
           markeredgecolor='#2878B5', label='Predicted but not detected'),
    Line2D([0], [0], marker='*', linestyle='none', markersize=12,
           markerfacecolor='#D73027', markeredgecolor='black', label='Detected event'),
    Line2D([0], [0], marker='*', linestyle='none', markersize=12,
           markerfacecolor='#F2B701', markeredgecolor='black', label='21CMA E08'),
]
regional_axis.legend(handles=regional_legend, loc='lower left', fontsize=8.3)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
figure_png = GROUND_TRACK_OUTPUT_STEM.with_suffix('.png')
figure_pdf = GROUND_TRACK_OUTPUT_STEM.with_suffix('.pdf')
fig.savefig(figure_png, dpi=300, bbox_inches='tight')
fig.savefig(figure_pdf, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(figure_png)))
print('Saved:', figure_png)
print('Saved:', figure_pdf)
print('Coastline source:', COASTLINE_GEOJSON)
print('Administrative boundaries drawn: no')


## 4. 事件附近的标准TLE缺口与SupGP轨道变化

本节把STARLINK-32838的标准公共TLE和CelesTrak `SpaceX-E` SupGP放在同一时间轴上，只研究2025-04-10至2025-04-20的事件附近窗口。图中不跨越缺口连接标准TLE，以免把“无标准TLE发布”误画成连续轨迹。

四个panel依次显示：

1. 由平均运动换算的平均轨道高度，以及包围Event 1和Event 2的标准TLE缺口；
2. 相邻SupGP平均高度的有限差分变化率，用于显示由下降转为抬升的时刻；
3. SupGP中的`MEAN_MOTION_DOT`和`BSTAR`。CSV中的`MEAN_MOTION_DOT`沿用TLE格式的$\dot n/2$，图中乘2后显示近似$\dot n$；
4. SupGP对SpaceX源星历的SGP4拟合RMS。

`BSTAR`、`MEAN_MOTION_DOT`和RMS均为辅助诊断量。负`BSTAR`不能解释为真实负大气阻力，RMS也不是位置协方差；判断轨道抬升首先依据`MEAN_MOTION`及其换算高度的实际趋势。


In [ ]:
# 22.1 同图比较标准TLE、SupGP以及事件附近的轨道变化诊断量
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import Image, display

ORBIT_ANALYSIS_OUTPUT_DIR = OUTPUT_DIR / 'orbit_evolution'
ORBIT_ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_TLE_HISTORY_CSV = ORBIT_ANALYSIS_OUTPUT_DIR / 'starlink_32838_orbit_history.csv'
SUPGP_HISTORY_CSV = DATA_DIR / 'orbital' / 'STARLINK-32838_SpaceX-E_SupGP.csv'
if not TARGET_TLE_HISTORY_CSV.is_file():
    subprocess.run([sys.executable, str(PROJECT_ROOT / '4_orbital_context/scripts/plot_starlink_orbit_history.py'), '--input', str(DATA_DIR / 'orbital/STARLINK-32838_full_history.tle'), '--output-dir', str(ORBIT_ANALYSIS_OUTPUT_DIR)], check=True)
EVENT1_PREAMBLE_UTC = pd.Timestamp('2025-04-15T14:00:00.412698Z')
EVENT2_PREAMBLE_UTC = pd.Timestamp('2025-04-15T14:01:40.399723Z')
PLOT_START_UTC = pd.Timestamp('2025-04-10T00:00:00Z')
PLOT_STOP_UTC = pd.Timestamp('2025-04-20T23:59:59Z')
WGS72_MU_KM3_S2 = 398600.8
WGS72_EARTH_RADIUS_KM = 6378.135

for required_path in (TARGET_TLE_HISTORY_CSV, SUPGP_HISTORY_CSV):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

target_tle = pd.read_csv(TARGET_TLE_HISTORY_CSV)
target_tle['epoch_utc'] = pd.to_datetime(target_tle['epoch_utc'], utc=True)
target_tle_window = target_tle.loc[
    target_tle['epoch_utc'].between(PLOT_START_UTC, PLOT_STOP_UTC)
].copy().sort_values('epoch_utc')

supgp = pd.read_csv(SUPGP_HISTORY_CSV)
supgp['epoch_utc'] = pd.to_datetime(supgp['EPOCH'], utc=True)
if set(supgp['NORAD_CAT_ID'].astype(int)) != {62827}:
    raise ValueError('SupGP文件并非只包含NORAD 62827。')
mean_motion_rad_s = (
    supgp['MEAN_MOTION'].astype(float).to_numpy() * 2.0 * np.pi / 86400.0
)
supgp['mean_altitude_km'] = (
    (WGS72_MU_KM3_S2 / mean_motion_rad_s**2.0) ** (1.0 / 3.0)
    - WGS72_EARTH_RADIUS_KM
)
supgp = supgp.sort_values('epoch_utc').reset_index(drop=True)
supgp['interval_hours'] = supgp['epoch_utc'].diff().dt.total_seconds() / 3600.0
supgp['altitude_rate_km_per_day'] = (
    supgp['mean_altitude_km'].diff()
    / (supgp['interval_hours'] / 24.0)
)
supgp['rate_midpoint_utc'] = (
    supgp['epoch_utc'].shift(1)
    + (supgp['epoch_utc'] - supgp['epoch_utc'].shift(1)) / 2
)
supgp['mean_motion_derivative_rev_day2'] = (
    2.0 * supgp['MEAN_MOTION_DOT'].astype(float)
)

tle_before = target_tle.loc[target_tle['epoch_utc'] <= EVENT1_PREAMBLE_UTC].iloc[-1]
tle_after = target_tle.loc[target_tle['epoch_utc'] >= EVENT1_PREAMBLE_UTC].iloc[0]
tle_gap_hours = (tle_after['epoch_utc'] - tle_before['epoch_utc']).total_seconds() / 3600.0

derived_csv = ORBIT_ANALYSIS_OUTPUT_DIR / 'event_centered_tle_supgp_diagnostics.csv'
supgp.to_csv(derived_csv, index=False)

fig, axes = plt.subplots(4, 1, figsize=(13.5, 11.0), sharex=True, constrained_layout=True)

def add_common_time_context(axis):
    axis.axvspan(
        tle_before['epoch_utc'], tle_after['epoch_utc'],
        color='0.75', alpha=0.18, zorder=0,
    )
    axis.axvspan(
        EVENT1_PREAMBLE_UTC, EVENT2_PREAMBLE_UTC,
        color='#D73027', alpha=0.18, zorder=1,
    )
    axis.grid(axis='y', color='0.87', linewidth=0.8)
    axis.tick_params(axis='both', which='major', labelsize=11)

ax = axes[0]
add_common_time_context(ax)
ax.scatter(
    target_tle_window['epoch_utc'], target_tle_window['mean_altitude_km'],
    s=42, color='#2878B5', edgecolor='white', linewidth=0.5,
    label='TLE', zorder=4,
)
ax.plot(
    supgp['epoch_utc'], supgp['mean_altitude_km'], '-o',
    color='#E67E22', linewidth=2.0, markersize=4.2,
    label='SpaceX-E SupGP', zorder=3,
)
ax.set_ylabel('Mean altitude (km)', fontsize=11)
ax.set_title('(a) Public TLE gap and continuous SpaceX-E SupGP coverage')
ax.legend(loc='lower right', frameon=True)

ax = axes[1]
add_common_time_context(ax)
ax.axhline(0.0, color='0.25', linewidth=1.0)
rate_rows = supgp.loc[
    supgp['rate_midpoint_utc'].notna()
    & supgp['altitude_rate_km_per_day'].notna()
]
ax.plot(
    rate_rows['rate_midpoint_utc'], rate_rows['altitude_rate_km_per_day'],
    '-o', color='#2A9D8F', linewidth=1.8, markersize=4.0,
)
ax.set_ylabel(r'$\Delta h/\Delta t$ (km day$^{-1}$)', fontsize=11)
ax.set_title('(b) Finite-difference change rate of SupGP mean altitude')

ax = axes[2]
add_common_time_context(ax)
motion_line = ax.plot(
    supgp['epoch_utc'], supgp['mean_motion_derivative_rev_day2'],
    '-o', color='#7B2CBF', linewidth=1.8, markersize=3.8,
    label=r'Fitted mean-motion derivative $\dot{n}$',
)[0]
ax.set_ylabel(r'$\dot{n}$ (rev day$^{-2}$)', color='black', fontsize=11)
ax.tick_params(axis='y', colors='black')
bstar_axis = ax.twinx()
bstar_line = bstar_axis.plot(
    supgp['epoch_utc'], supgp['BSTAR'].astype(float),
    '-s', color='#C44E52', linewidth=1.6, markersize=3.5, label='BSTAR',
)[0]
bstar_axis.set_ylabel(r'BSTAR (Earth radii$^{-1}$)', color='black', fontsize=11)
bstar_axis.tick_params(axis='y', colors='black', labelsize=11)
motion_limit = 1.08 * np.nanmax(
    np.abs(supgp['mean_motion_derivative_rev_day2'].to_numpy(float))
)
bstar_limit = 1.08 * np.nanmax(np.abs(supgp['BSTAR'].to_numpy(float)))
ax.set_ylim(-motion_limit, motion_limit)
bstar_axis.set_ylim(-bstar_limit, bstar_limit)
ax.axhline(0.0, color='0.30', linewidth=0.9, zorder=1)
for spine in ax.spines.values():
    spine.set_color('black')
for spine in bstar_axis.spines.values():
    spine.set_color('black')
ax.set_title('(c) SupGP fitted mean-motion derivative and BSTAR')
ax.legend(handles=[motion_line, bstar_line], loc='upper right', fontsize=8.7)

ax = axes[3]
add_common_time_context(ax)
ax.plot(
    supgp['epoch_utc'], supgp['RMS'].astype(float), '-o',
    color='#264653', linewidth=1.8, markersize=4.0,
)
ax.set_ylabel('Fit RMS (km)', fontsize=11)
ax.set_title('(d) SupGP RMS')
ax.set_xlabel('UTC date in 2025', fontsize=11)
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

figure_png = ORBIT_ANALYSIS_OUTPUT_DIR / 'event_centered_tle_supgp_orbit_evolution.png'
figure_pdf = figure_png.with_suffix('.pdf')
fig.savefig(figure_png, dpi=300, bbox_inches='tight')
fig.savefig(figure_pdf, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(figure_png)))
print(f'Standard TLE gap: {tle_gap_hours:.6f} h')
print('Last TLE before Event 1:', tle_before['epoch_utc'])
print('First TLE after Event 1:', tle_after['epoch_utc'])
print('Saved:', figure_png)
print('Saved:', figure_pdf)
print('Saved:', derived_csv)


## 5. Event 1实际接收时刻的Group 11-6高度分布

本节统一采用acquisition metadata的Event 1 LoRa前导开始时刻`2025-04-15 14:00:00.412698 UTC`，为Group 11-6每颗卫星选择距离该时刻最近的一条标准TLE，并比较其平均轨道高度。这里不使用$W=100$ s盲筛所得的最佳时间偏移：该偏移是旧TLE状态匹配参数，不是观测UTC。



In [ ]:
# 23.1 绘制Event 1时刻同批23颗卫星的最近TLE平均高度
GROUP_HISTORY_CSV = ORBIT_ANALYSIS_OUTPUT_DIR / 'group11_6_orbit_history.csv'
if not GROUP_HISTORY_CSV.is_file():
    subprocess.run([sys.executable, str(PROJECT_ROOT / '4_orbital_context/scripts/compare_group11_6_orbits.py'), '--input', str(DATA_DIR / 'orbital/STARLINK_Group11-6_full_history.tle'), '--output-dir', str(ORBIT_ANALYSIS_OUTPUT_DIR)], check=True)
TARGET_NORAD_ID = 62827
PAYLOAD_INSTANTANEOUS_GEODETIC_ALTITUDE_KM = 323.58690316224056

group_history = pd.read_csv(GROUP_HISTORY_CSV)
group_history['epoch_utc'] = pd.to_datetime(
    group_history['epoch_utc'], format='mixed', utc=True
)
group_history['time_offset_s'] = (
    group_history['epoch_utc'] - EVENT1_PREAMBLE_UTC
).dt.total_seconds()
nearest_indices = (
    group_history.assign(abs_time_offset_s=group_history['time_offset_s'].abs())
    .groupby('norad_id')['abs_time_offset_s']
    .idxmin()
)
event_snapshot = (
    group_history.loc[nearest_indices]
    .copy()
    .sort_values('norad_id')
    .reset_index(drop=True)
)
event_snapshot['tle_age_hours'] = event_snapshot['time_offset_s'] / 3600.0
event_snapshot['is_target'] = event_snapshot['norad_id'].astype(int) == TARGET_NORAD_ID
if len(event_snapshot) != 23 or int(event_snapshot['is_target'].sum()) != 1:
    raise ValueError('Group 11-6应包含23颗卫星且目标只能有一颗。')

peer_snapshot = event_snapshot.loc[~event_snapshot['is_target']]
target_snapshot = event_snapshot.loc[event_snapshot['is_target']].iloc[0]
peer_median_km = float(peer_snapshot['mean_altitude_km'].median())
target_offset_from_peer_km = float(target_snapshot['mean_altitude_km'] - peer_median_km)

snapshot_csv = ORBIT_ANALYSIS_OUTPUT_DIR / 'group11_6_event1_altitude_snapshot.csv'
event_snapshot.to_csv(snapshot_csv, index=False)

fig, ax = plt.subplots(figsize=(14.5, 6.8), constrained_layout=True)
x = np.arange(len(event_snapshot))
peer_mask = ~event_snapshot['is_target'].to_numpy(bool)
target_index = int(np.flatnonzero(~peer_mask)[0])
ax.scatter(
    x[peer_mask], event_snapshot.loc[peer_mask, 'mean_altitude_km'],
    s=65, color='#4C78A8', edgecolor='white', linewidth=0.6,
    label='other Group 11-6 spacecraft', zorder=3,
)
ax.scatter(
    [target_index], [target_snapshot['mean_altitude_km']],
    s=230, marker='*', color='#D73027', edgecolor='black', linewidth=0.8,
    label='STARLINK-32838 nearest-TLE mean altitude', zorder=6,
)
ax.axhline(
    PAYLOAD_INSTANTANEOUS_GEODETIC_ALTITUDE_KM,
    color='#F2B701', linestyle='--', linewidth=1.8,
    label=(
        'STARLINK-32838 payload instantaneous geodetic altitude '
        f'= {PAYLOAD_INSTANTANEOUS_GEODETIC_ALTITUDE_KM:.2f} km'
    ),
    zorder=4,
)
ax.set_xticks(x)
ax.set_xticklabels(event_snapshot['norad_id'].astype(int).astype(str), rotation=55, ha='right')
ax.set_xlabel('NORAD catalogue ID')
ax.set_ylabel('Nearest-TLE mean altitude (km)')
ax.set_ylim(300.0, 460.0)
ax.set_title(
    'Starlink Group 11-6 altitude context at Event 1 preamble start\n'
    + EVENT1_PREAMBLE_UTC.strftime('%Y-%m-%d %H:%M:%S.%f UTC')
)
ax.legend(loc='lower right', fontsize=8.7)

snapshot_png = ORBIT_ANALYSIS_OUTPUT_DIR / 'group11_6_event1_altitude_snapshot.png'
snapshot_pdf = snapshot_png.with_suffix('.pdf')
fig.savefig(snapshot_png, dpi=300, bbox_inches='tight')
fig.savefig(snapshot_pdf, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(snapshot_png)))
print(f'Peer altitude range: {peer_snapshot.mean_altitude_km.min():.3f} to '
      f'{peer_snapshot.mean_altitude_km.max():.3f} km')
print(f'Peer median altitude: {peer_median_km:.3f} km')
print(f'Target nearest-TLE mean altitude: {target_snapshot.mean_altitude_km:.3f} km')
print(f'Target nearest-TLE age: {target_snapshot.tle_age_hours:.3f} h')
print(f'Target minus peer median: {target_offset_from_peer_km:.3f} km')
print('Saved:', snapshot_png)
print('Saved:', snapshot_pdf)
print('Saved:', snapshot_csv)


## 6. Group 11-6从初始编目到STARLINK-32838衰减再入的轨道高度

本节用Group 11-6的完整TLE历史展示STARLINK-32838相对同批卫星的长期轨道演化。其他22颗卫星以浅灰细线表示，并在每日共同时间网格上计算中位数和10%–90%分位包络；STARLINK-32838使用红线突出。下方面板给出目标相对同批中位高度的差值。

图中的TLE历元之间连线只用于展示长期演化趋势，不代表连续测轨；Event 1竖线表示VHF观测时刻，`decay`竖线表示编目衰减日期。该图可以证明目标的轨道演化相对同批显著异常，但不能单独确定异常原因或把后续衰减倒因为2025年4月的VHF发射。


In [ ]:
# 24.1 绘制Group 11-6完整TLE高度历史、同批中位高度及目标相对偏差
TARGET_DECAY_UTC = pd.Timestamp('2026-01-05T00:00:00Z')
history_until_decay = group_history.loc[
    group_history['epoch_utc'] <= TARGET_DECAY_UTC
].copy().sort_values(['norad_id', 'epoch_utc'])
COMMON_HISTORY_START_UTC = (
    history_until_decay.groupby('norad_id')['epoch_utc'].min().max()
)
history_for_plot = history_until_decay.loc[
    history_until_decay['epoch_utc'] >= COMMON_HISTORY_START_UTC
].copy()
peer_ids = sorted(set(history_for_plot['norad_id'].astype(int)) - {TARGET_NORAD_ID})
target_history = history_for_plot.loc[
    history_for_plot['norad_id'].astype(int) == TARGET_NORAD_ID
].copy()

daily_utc = pd.date_range(
    COMMON_HISTORY_START_UTC,
    TARGET_DECAY_UTC - pd.Timedelta(seconds=1), freq='1D',
)
daily_seconds = daily_utc.astype('int64').to_numpy(dtype=float) / 1e9
peer_daily_rows = []
for norad_id in peer_ids:
    rows = history_until_decay.loc[
        history_until_decay['norad_id'].astype(int) == norad_id
    ]
    source_seconds = rows['epoch_utc'].astype('int64').to_numpy(dtype=float) / 1e9
    source_altitude = rows['mean_altitude_km'].to_numpy(float)
    interpolated = np.interp(daily_seconds, source_seconds, source_altitude)
    interpolated[(daily_seconds < source_seconds.min()) | (daily_seconds > source_seconds.max())] = np.nan
    peer_daily_rows.append(interpolated)
peer_daily = np.asarray(peer_daily_rows, dtype=float)
peer_q10 = np.nanquantile(peer_daily, 0.10, axis=0)
peer_median = np.nanmedian(peer_daily, axis=0)
peer_q90 = np.nanquantile(peer_daily, 0.90, axis=0)

target_history_full = history_until_decay.loc[
    history_until_decay['norad_id'].astype(int) == TARGET_NORAD_ID
]
target_seconds = target_history_full['epoch_utc'].astype('int64').to_numpy(dtype=float) / 1e9
target_altitude = target_history_full['mean_altitude_km'].to_numpy(float)
target_daily = np.interp(daily_seconds, target_seconds, target_altitude)
target_daily[(daily_seconds < target_seconds.min()) | (daily_seconds > target_seconds.max())] = np.nan
target_minus_peer_median = target_daily - peer_median

daily_context = pd.DataFrame({
    'utc': daily_utc,
    'peer_q10_mean_altitude_km': peer_q10,
    'peer_median_mean_altitude_km': peer_median,
    'peer_q90_mean_altitude_km': peer_q90,
    'target_interpolated_mean_altitude_km': target_daily,
    'target_minus_peer_median_km': target_minus_peer_median,
})
lifecycle_csv = ORBIT_ANALYSIS_OUTPUT_DIR / 'group11_6_daily_lifecycle_context.csv'
daily_context.to_csv(lifecycle_csv, index=False)

fig, ax = plt.subplots(figsize=(6.8, 4.8), constrained_layout=True)
for norad_id in peer_ids:
    rows = history_for_plot.loc[history_for_plot['norad_id'].astype(int) == norad_id]
    ax.plot(
        rows['epoch_utc'], rows['mean_altitude_km'],
        color='0.68', linewidth=0.65, alpha=0.42, zorder=1,
    )
ax.plot(
    daily_utc, peer_median, color='#1F4E79', linewidth=2.1,
    label='Peer median', zorder=3,
)
ax.plot(
    target_history['epoch_utc'], target_history['mean_altitude_km'],
    color='#D73027', linewidth=2.5, label='STARLINK-32838', zorder=5,
)
ax.set_ylabel('Mean altitude (km)', fontsize=12.5)
ax.set_xlabel('UTC date', fontsize=12.5)
ax.set_ylim(220.0, 500.0)
ax.grid(axis='y', color='0.88', linewidth=0.8)
ax.tick_params(axis='both', which='major', labelsize=11.5)
ax.legend(loc='lower left', fontsize=11.5)
ax.axvline(
    EVENT1_PREAMBLE_UTC, color='#F2B701', linestyle='--', linewidth=1.6,
)
ax.axvline(
    TARGET_DECAY_UTC, color='black', linestyle=':', linewidth=1.5,
)
ax.text(
    EVENT1_PREAMBLE_UTC, 493.0, ' VHF events',
    ha='left', va='top', fontsize=11.5, color='#8A6500',
)
ax.text(
    TARGET_DECAY_UTC, 493.0, ' Decay',
    ha='right', va='top', fontsize=11.5, color='black',
)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(
    ax.get_xticklabels(), rotation=35, ha='right',
)

lifecycle_png = ORBIT_ANALYSIS_OUTPUT_DIR / 'group11_6_full_lifecycle_altitude.png'
lifecycle_pdf = lifecycle_png.with_suffix('.pdf')
fig.savefig(lifecycle_png, dpi=300, bbox_inches='tight')
fig.savefig(lifecycle_pdf, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(lifecycle_png)))
print('Common history start:', COMMON_HISTORY_START_UTC)
print('Target final TLE epoch:', target_history['epoch_utc'].max())
print('Target catalogue decay date:', TARGET_DECAY_UTC)
print('Saved:', lifecycle_png)
print('Saved:', lifecycle_pdf)
print('Saved:', lifecycle_csv)


## 7. Missing 1和Missing 2相对Event 2的路径衰减

本节复用第2节确定的四个几何可见机会及其SupGP轨道几何。高度角、方位角和斜距统一取各候选机会预测起点的瞬时值。根据自由空间传播的$P\propto R^{-2}$关系，Missing 1和Missing 2相对Event 2的额外路径衰减计算为

$$\Delta L_i=20\log_{10}\left(\frac{R_i}{R_{\rm Event\,2}}\right).$$

该结果只表示斜距平方关系导致的相对传播衰减，不包含E08方向增益、极化失配、地形遮挡或卫星发射功率变化。

In [ ]:
# 25.1 用四个可见机会的预测起点斜距，计算相对Event 2的路径衰减
PATH_LOSS_OUTPUT_DIR = OUTPUT_DIR / 'path_loss'
PATH_LOSS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 严格沿用第20节的判据：-1、0、+1 s三个采样点均高于几何地平线。
visible_keys = (
    supgp_cadence_visibility_window_summary.loc[
        supgp_cadence_visibility_window_summary['sampled_horizon_status']
        == 'all_three_above_horizon',
        ['dataset', 'cycle_index_from_event1'],
    ]
    .drop_duplicates()
)
visible_opportunities = (
    supgp_cadence_visibility.merge(
        visible_keys, on=['dataset', 'cycle_index_from_event1'], how='inner'
    )
    .sort_values('predicted_utc')
    .reset_index(drop=True)
)
if len(visible_opportunities) != 4:
    raise ValueError(
        f'Expected four all-three-above-horizon opportunities; found {len(visible_opportunities)}'
    )

# 保留两个实测事件标签，其后两个机会按时间命名为Missing 1和Missing 2。
opportunity_labels = []
missing_index = 0
for observed_label in visible_opportunities['observed_event_id']:
    if pd.notna(observed_label) and str(observed_label).strip():
        opportunity_labels.append(str(observed_label))
    else:
        missing_index += 1
        opportunity_labels.append(f'Missing {missing_index}')
visible_opportunities['event_id'] = opportunity_labels
expected_labels = ['Event 1', 'Event 2', 'Missing 1', 'Missing 2']
if visible_opportunities['event_id'].tolist() != expected_labels:
    raise ValueError(
        'Unexpected visible-opportunity order: '
        f"{visible_opportunities['event_id'].tolist()}"
    )

central_geometry = visible_opportunities[[
    'event_id', 'predicted_utc', 'azimuth_deg', 'elevation_deg',
    'slant_range_km', 'supgp_epoch_utc',
]].copy()
central_geometry = central_geometry.rename(
    columns={'predicted_utc': 'candidate_start_utc'}
)
reference_range_km = float(
    central_geometry.loc[
        central_geometry['event_id'] == 'Event 2', 'slant_range_km'
    ].iloc[0]
)
relative_path_loss = central_geometry.loc[
    central_geometry['event_id'].isin(['Missing 1', 'Missing 2']),
    ['event_id', 'slant_range_km'],
].copy()
relative_path_loss.insert(1, 'reference_event_id', 'Event 2')
relative_path_loss['reference_slant_range_km'] = reference_range_km
relative_path_loss['path_attenuation_relative_to_event2_db'] = (
    20.0 * np.log10(
        relative_path_loss['slant_range_km'] / reference_range_km
    )
)

central_geometry_path = (
    PATH_LOSS_OUTPUT_DIR / 'four_visible_opportunity_predicted_start_geometry.csv'
)
relative_path_loss_path = (
    PATH_LOSS_OUTPUT_DIR / 'missing_events_relative_path_loss.csv'
)
central_geometry.to_csv(
    central_geometry_path, index=False, float_format='%.12f'
)
relative_path_loss.to_csv(
    relative_path_loss_path, index=False, float_format='%.12e'
)

display(central_geometry)
display(relative_path_loss)
print('Saved:', central_geometry_path)
print('Saved:', relative_path_loss_path)
